# Exporta.co — Pipeline de Datos V3
**Maestría en Ciencia de Datos | Visualización y Storytelling con Datos**  
Responsable: Camilo José Beltrán Rojas

### Cambios principales vs V2
- **Modo prueba**: `SAMPLE_N` limita a N filas por CSV para iterar rápido  
- **Códigos IMPO**: diccionario DANE para 2011-2024, pycountry ISO para 2025+  
- **Procesamiento año x año**: libera RAM entre años (compatible con Colab free)  
- **Cache Parquet**: segunda ejecución carga en <2 min sin reprocesar ZIPs

import glob, os
for f in glob.glob(str(CLEAN_DIR / 'expo_*.*')) + glob.glob(str(CLEAN_DIR / 'impo_*.*')):
    os.remove(f)
    print(f'Borrado: {Path(f).name}')
print('Cache limpio. Re-ejecutar S8.')

---
## Sección 1 — Configuración de entorno y rutas

In [96]:
from pathlib import Path
import subprocess, sys

# -- MODO DE EJECUCION ---
MODO = 'local'           # Cambiar a 'colab' en Google Colab

# -- MODO PRUEBA ---
# None = procesar TODO.  Entero (ej: 1000) = top N filas por CSV
SAMPLE_N = None

if MODO == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR  = Path('/content/drive/MyDrive/exporta_co')
    RAW_DIR    = DRIVE_DIR / 'raw'
    CLEAN_DIR  = DRIVE_DIR / 'clean'
    REPO_URL  = 'https://github.com/CamiloJose90/Exporta_Co.git'
    REPO_DIR  = Path('/content/Exporta_Co')
    if not REPO_DIR.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull'], check=True)
    OUTPUT_DIR = REPO_DIR / 'data'
else:
    RAW_DIR    = Path(r'C:\Users\camil\Documentos\MIAD\Visualización y storytelling\Proyecto')
    CLEAN_DIR  = RAW_DIR / 'clean'
    OUTPUT_DIR = RAW_DIR / 'json_output'

CLEAN_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
BASE_DIR = RAW_DIR

modo_txt = f'PRUEBA (top {SAMPLE_N} filas/CSV)' if SAMPLE_N else 'COMPLETO'
print(f'Modo: {modo_txt}')
print(f'RAW_DIR   : {RAW_DIR}')
print(f'CLEAN_DIR : {CLEAN_DIR}')
print(f'OUTPUT_DIR: {OUTPUT_DIR}')

Modo: COMPLETO
RAW_DIR   : C:\Users\camil\Documentos\MIAD\Visualización y storytelling\Proyecto
CLEAN_DIR : C:\Users\camil\Documentos\MIAD\Visualización y storytelling\Proyecto\clean
OUTPUT_DIR: C:\Users\camil\Documentos\MIAD\Visualización y storytelling\Proyecto\json_output


---
## Sección 2 — Imports

In [97]:
import importlib
if importlib.util.find_spec('pycountry') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'pycountry', '-q'])

# Fix compatibilidad pyarrow/pandas
try:
    import pyarrow
    # Forzar reset del extension type si ya esta registrado
    try:
        pyarrow.unregister_extension_type("arrow.py_extension_type")
    except Exception:
        pass
except ImportError:
    pass

import pandas as pd
import numpy as np
import pycountry
import json, zipfile, io, re, gc
from pathlib import Path
from collections import defaultdict

print(f'pandas {pd.__version__} | numpy {np.__version__}')
try:
    print(f'pyarrow {pyarrow.__version__}')
except:
    print('pyarrow: no disponible')

pandas 3.0.1 | numpy 2.4.3
pyarrow 23.0.1


---
## Sección 3 — Configuración y diccionarios
Centraliza parámetros, mapas de columnas y diccionarios de referencia.  
**Cambio clave V3**: diccionario `CODIGOS_PAIS_IMPO` para 2011-2024 (códigos DANE internos).

In [98]:
# Rango de anios a procesar
ANIOS = list(range(2011, 2026))

# -- Mapeo columnas EXPO --
COLUMNAS_EXPO = {
    '\xef\xbb\xbffech': 'fech', 'fech': 'fech',
    '\ufefffech': 'fech',
    'cod_pai4': 'pais',
    'posar':    'subpartida',
    'fobdol':   'fob_usd',
    'pnk':      'kg_neto',
}

# -- Mapeo columnas IMPO --
COLUMNAS_IMPO = {
    '\xef\xbb\xbffech': 'fech', 'fech': 'fech',
    '\ufefffech': 'fech',
    'paisgen':  'pais',
    'naban':    'subpartida',
    'vacid':    'cif_usd',
    'pnk':      'kg_neto',
}

# -- Sectores OMC desde capitulo HS --
_SECTOR_BREAKS = [0, 24, 27, 97]
_SECTOR_LABELS = [
    'Agropecuarios, alimentos y bebidas',
    'Combustibles e industrias extractivas',
    'Manufacturas',
]

# -- Mes en espanol --
MESES_ES = {
    'enero':1, 'febrero':2, 'marzo':3, 'abril':4,
    'mayo':5, 'junio':6, 'julio':7, 'agosto':8,
    'septiembre':9, 'octubre':10, 'noviembre':11, 'diciembre':12,
}

# -- UMBRAL para codigos de pais en IMPO --
IMPO_ISO_DESDE = 2025

# -- Diccionario OFICIAL DANE/DIAN de paises para IMPO 2011-2024 --
# Fuente: diccionario de variables DANE, campo PAISGEN
# Zonas francas (911-928) y no declarados (999) se excluyen.
CODIGOS_PAIS_IMPO = {
    # A
    13:'AFG', 17:'ALB', 23:'DEU', 26:'ARM', 27:'ABW', 29:'BIH', 31:'BFA', 37:'AND',
    40:'AGO', 41:'AIA', 43:'ATG', 47:'ANT', 53:'SAU', 59:'DZA', 63:'ARG', 69:'AUS',
    72:'AUT', 74:'AZE',
    # B
    77:'BHS', 80:'BHR', 81:'BGD', 83:'BRB', 87:'BEL', 88:'BLZ', 90:'BMU', 91:'BLR',
    93:'MMR', 97:'BOL', 101:'BWA', 105:'BRA', 108:'BRN', 111:'BGR', 115:'BDI', 119:'BTN',
    # C
    127:'CPV', 129:'XKX', 130:'MNE', 137:'CYM', 141:'KHM', 145:'CMR', 149:'CAN',
    159:'VAT', 165:'CCK', 169:'COL', 173:'COM', 177:'COG', 183:'COK', 187:'PRK',
    190:'KOR', 193:'CIV', 196:'CRI', 198:'HRV', 199:'CUB', 200:'BES', 203:'TCD',
    # D-E
    211:'CHL', 215:'CHN', 218:'TWN', 221:'CYP', 229:'BEN', 232:'DNK', 235:'DMA',
    239:'ECU', 240:'EGY', 242:'SLV', 243:'ERI', 244:'ARE', 245:'ESP', 246:'SVK',
    247:'SVN', 249:'USA', 251:'EST', 253:'ETH', 259:'FRO',
    # F-G
    267:'PHL', 271:'FIN', 275:'FRA', 281:'GAB', 285:'GMB', 287:'GEO', 289:'GHA',
    293:'GIB', 297:'GRD', 301:'GRC', 305:'GRL', 309:'GLP', 313:'GUM', 317:'GTM',
    325:'GUF', 329:'GIN', 331:'GNQ', 334:'GNB', 337:'GUY',
    # H-K
    341:'HTI', 345:'HND', 351:'HKG', 355:'HUN', 361:'IND', 365:'IDN', 369:'IRQ',
    372:'IRN', 375:'IRL', 379:'ISL', 383:'ISR', 386:'ITA', 391:'JAM', 399:'JPN',
    403:'JOR', 406:'KAZ', 410:'KEN', 411:'KIR', 412:'KGZ', 413:'KWT',
    # L-M
    420:'LAO', 426:'LSO', 429:'LVA', 431:'LBN', 434:'LBR', 438:'LBY', 440:'LIE',
    443:'LTU', 445:'LUX', 447:'MAC', 448:'MKD', 450:'MDG', 455:'MYS', 458:'MWI',
    461:'MDV', 464:'MLI', 467:'MLT', 469:'MNP', 472:'MHL', 474:'MAR', 477:'MTQ',
    485:'MUS', 488:'MRT', 493:'MEX', 494:'FSM', 496:'MDA', 497:'MNG', 498:'MCO',
    501:'MSR', 505:'MOZ', 507:'NAM', 508:'NRU',
    # N-P
    511:'CXR', 517:'NPL', 521:'NIC', 525:'NER', 528:'NGA', 531:'NIU', 535:'NFK',
    538:'NOR', 542:'NCL', 545:'PNG', 548:'NZL', 551:'VUT', 556:'OMN', 566:'UMI',
    573:'NLD', 576:'PAK', 578:'PLW', 579:'PSE', 580:'PAN', 586:'PRY', 589:'PER',
    593:'PCN', 599:'PYF', 603:'POL', 607:'PRT', 611:'PRI', 618:'QAT',
    # R-S
    628:'GBR', 631:'SXM', 633:'CUW', 640:'CAF', 644:'CZE', 647:'DOM', 651:'SSP',
    654:'SSD', 658:'SSD', 660:'REU', 665:'ZWE', 670:'ROU', 675:'RWA', 676:'RUS',
    677:'SLB', 685:'ESH', 687:'WSM', 688:'SSD', 690:'ASM', 695:'KNA', 697:'SMR',
    700:'SPM', 705:'VCT', 710:'SHN', 715:'LCA', 720:'STP', 728:'SEN', 729:'SRB',
    731:'SYC', 735:'SLE', 741:'SGP', 744:'SYR', 748:'SOM', 750:'LKA', 756:'ZAF',
    759:'SDN', 764:'SWE', 767:'CHE', 770:'SUR', 773:'SWZ', 774:'TJK', 776:'THA',
    # T-Z
    780:'TZA', 783:'DJI', 787:'IOT', 788:'TLS', 800:'TGO', 805:'TKL', 810:'TON',
    815:'TTO', 820:'TUN', 823:'TCA', 825:'TKM', 827:'TUR', 828:'TUV', 830:'UKR',
    833:'UGA', 845:'URY', 847:'UZB', 850:'VEN', 855:'VNM', 863:'VGB', 866:'VIR',
    870:'FJI', 875:'WLF', 880:'YEM', 885:'SRB', 888:'COD', 890:'ZMB',
}


print('Configuracion cargada')
print(f'Anios: {ANIOS[0]}-{ANIOS[-1]}')
print(f'IMPO ISO desde: {IMPO_ISO_DESDE}')
print(f'Codigos DANE mapeados: {len(CODIGOS_PAIS_IMPO)} paises')

Configuracion cargada
Anios: 2011-2025
IMPO ISO desde: 2025
Codigos DANE mapeados: 241 paises


---
## Sección 4 — Descubrimiento de archivos

In [99]:
PATRON = re.compile(r'^(expo|impo)_(\d{4})(?:_(\d))?\.zip$', re.IGNORECASE)
archivos = {'expo': defaultdict(list), 'impo': defaultdict(list)}

for f in sorted(BASE_DIR.glob('*.zip')):
    m = PATRON.match(f.name)
    if m:
        tipo = m.group(1).lower()
        anio = int(m.group(2))
        archivos[tipo][anio].append(f)

for tipo in ('expo', 'impo'):
    anios_rango   = sorted(a for a in archivos[tipo] if a in ANIOS)
    anios_faltant = [a for a in ANIOS if a not in archivos[tipo]]
    print(f'\n{tipo.upper()} -- {len(anios_rango)} anio(s) en rango')
    for anio in anios_rango:
        partes = archivos[tipo][anio]
        tag = f' [{len(partes)} partes]' if len(partes) > 1 else ''
        for p in partes:
            print(f'  {p.name}{tag}')
    if anios_faltant:
        print(f'  FALTANTES: {anios_faltant}')


EXPO -- 15 anio(s) en rango
  Expo_2011.zip
  Expo_2012.zip
  Expo_2013.zip
  Expo_2014.zip
  Expo_2015.zip
  Expo_2016.zip
  Expo_2017.zip
  Expo_2018.zip
  Expo_2019.zip
  Expo_2020.zip
  Expo_2021.zip
  Expo_2022.zip
  Expo_2023.zip
  Expo_2024.zip
  Expo_2025.zip

IMPO -- 15 anio(s) en rango
  Impo_2011.zip
  Impo_2012.zip
  Impo_2013.zip
  Impo_2014.zip
  Impo_2015.zip
  Impo_2016.zip
  Impo_2017.zip
  Impo_2018.zip
  Impo_2019.zip
  Impo_2020.zip
  Impo_2021_1.zip [2 partes]
  Impo_2021_2.zip [2 partes]
  Impo_2022_1.zip [2 partes]
  Impo_2022_2.zip [2 partes]
  Impo_2023.zip
  Impo_2024.zip
  Impo_2025_1.zip [2 partes]
  Impo_2025_2.zip [2 partes]


---
## Sección 5 — Funciones auxiliares

In [100]:
def normalizar_columnas(df, mapa):
    rename = {}
    for col in df.columns:
        clave = col.strip().lower().replace(' ', '_')
        if clave in mapa:
            rename[col] = mapa[clave]
        elif col.strip() in mapa:
            rename[col] = mapa[col.strip()]
    return df.rename(columns=rename)


def _limpiar_valor_monetario(serie):
    """Detecta formato europeo (4.885,70) vs estandar (4885.70) por fila."""
    s = serie.astype(str).str.strip()
    
    tiene_coma = s.str.contains(',', na=False)
    
    # Europea: quitar puntos de miles, coma -> punto decimal
    europea = (
        s.str.replace('.', '', regex=False)
        .str.replace(',', '.', regex=False)
        .pipe(pd.to_numeric, errors='coerce')
    )
    
    # Estandar: punto ya es decimal
    estandar = pd.to_numeric(s, errors='coerce')
    
    return europea.where(tiene_coma, estandar)


def limpiar_valor_expo(serie):
    return _limpiar_valor_monetario(serie)


def limpiar_valor_impo(serie):
    return _limpiar_valor_monetario(serie)


def derivar_sector(serie_subpartida):
    capitulo = pd.to_numeric(serie_subpartida.astype(str).str[:2], errors='coerce')
    return (
        pd.cut(capitulo, bins=_SECTOR_BREAKS, labels=_SECTOR_LABELS)
        .astype(str)
        .replace('nan', 'Otros sectores')
    )


def mes_desde_nombre(nombre_archivo):
    stem = Path(nombre_archivo).stem.lower().strip()
    primera = stem.split()[0] if ' ' in stem else stem
    return MESES_ES.get(primera, None)


def leer_csv_bytes(data_bytes, nrows=None):
    for encoding in ('latin-1', 'utf-8', 'utf-8-sig'):
        for sep in (',', ';', '\t'):
            try:
                df = pd.read_csv(
                    io.BytesIO(data_bytes), sep=sep, encoding=encoding,
                    low_memory=False, on_bad_lines='skip', nrows=nrows,
                )
                if len(df.columns) > 2:
                    return df.dropna(how='all')
            except Exception:
                continue
    return None


print('Funciones auxiliares listas')

Funciones auxiliares listas


---
## Sección 6 — Funciones de ingesta ZIP

In [101]:
def leer_zip_expo(ruta_zip):
    frames = []
    nrows = SAMPLE_N if SAMPLE_N else None
    try:
        with zipfile.ZipFile(ruta_zip, 'r') as zf:
            zips_mes = sorted(f for f in zf.namelist() if f.lower().endswith('.zip'))
            for nom_zip in zips_mes:
                try:
                    with zipfile.ZipFile(io.BytesIO(zf.read(nom_zip)), 'r') as zf_mes:
                        for csv_path in (f for f in zf_mes.namelist() if f.lower().endswith('.csv')):
                            df = leer_csv_bytes(zf_mes.read(csv_path), nrows=nrows)
                            if df is not None:
                                frames.append(normalizar_columnas(df, COLUMNAS_EXPO))
                except Exception as e:
                    print(f'    WARN {nom_zip}: {e}')
    except zipfile.BadZipFile:
        print(f'    ZIP corrupto: {ruta_zip.name}')
    return frames


def leer_zip_impo(ruta_zip):
    frames = []
    nrows = SAMPLE_N if SAMPLE_N else None
    try:
        with zipfile.ZipFile(ruta_zip, 'r') as zf:
            contenidos = zf.namelist()
            zips_mes = sorted(f for f in contenidos if f.lower().endswith('.zip'))
            csvs_dir = sorted(f for f in contenidos if f.lower().endswith('.csv'))

            if zips_mes:
                # Caso A: ZIPs mensuales (2011-2023, 2025)
                for nom_zip in zips_mes:
                    try:
                        with zipfile.ZipFile(io.BytesIO(zf.read(nom_zip)), 'r') as zf_mes:
                            for csv_path in (f for f in zf_mes.namelist() if f.lower().endswith('.csv')):
                                df = leer_csv_bytes(zf_mes.read(csv_path), nrows=nrows)
                                if df is not None:
                                    df = normalizar_columnas(df, COLUMNAS_IMPO)
                                    mes = mes_desde_nombre(csv_path)
                                    if mes is None:
                                        mes = mes_desde_nombre(nom_zip)
                                    if mes:
                                        df['mes_archivo'] = mes
                                    frames.append(df)
                    except Exception as e:
                        print(f'    WARN {nom_zip}: {e}')

            elif csvs_dir:
                # Caso B: CSVs en carpetas directas (2024)
                for csv_path in csvs_dir:
                    try:
                        df = leer_csv_bytes(zf.read(csv_path), nrows=nrows)
                        if df is not None:
                            df = normalizar_columnas(df, COLUMNAS_IMPO)
                            mes = mes_desde_nombre(csv_path)
                            if mes:
                                df['mes_archivo'] = mes
                            frames.append(df)
                    except Exception as e:
                        print(f'    WARN {csv_path}: {e}')
            else:
                print(f'    Sin CSVs ni ZIPs en {ruta_zip.name}')

    except zipfile.BadZipFile:
        print(f'    ZIP corrupto: {ruta_zip.name}')
    return frames


print('Funciones de ingesta listas')

Funciones de ingesta listas


---
## Sección 7 — Funciones de limpieza
Separadas por tipo. La clave es `limpiar_impo` que aplica el diccionario correcto según el año.

In [102]:
def limpiar_expo(df, anio):
    # Parsear fecha YYMM
    fech = pd.to_numeric(df['fech'], errors='coerce')
    df['anio'] = (2000 + fech // 100).astype('Int64')
    df['mes']  = (fech % 100).astype('Int64')
    
    # Pais: ISO alfa-3 directo en EXPO
    df['iso3'] = df['pais'].astype(str).str.strip().str.upper()
    
    # Valor FOB
    df['fob_usd'] = limpiar_valor_expo(df['fob_usd'])
    
    # Sector OMC
    df['sector'] = derivar_sector(df['subpartida'])
    
    # Filtrar
    mask = (
        df['fob_usd'].notna() & (df['fob_usd'] > 0) &
        df['iso3'].notna() & (df['iso3'] != '') & (df['iso3'] != 'NAN') &
        df['anio'].between(ANIOS[0], ANIOS[-1]) &
        df['mes'].between(1, 12)
    )
    cols = ['anio', 'mes', 'iso3', 'fob_usd', 'sector', 'subpartida', 'kg_neto']
    return df.loc[mask, [c for c in cols if c in df.columns]].copy()


def limpiar_impo(df, anio):
    # Parsear fecha YYMM
    fech = pd.to_numeric(df['fech'], errors='coerce')
    df['mes_fech'] = (fech % 100).astype('Int64')
    
    # Mes: preferir nombre de archivo
    if 'mes_archivo' in df.columns:
        df['mes'] = df['mes_archivo'].fillna(df['mes_fech']).astype('Int64')
    else:
        df['mes'] = df['mes_fech']
    
    # Anio: forzar desde el ZIP (mas confiable)
    df['anio'] = anio
    
    # Pais: resolver segun sistema del anio
    cod_num = pd.to_numeric(df['pais'], errors='coerce')
    if anio < IMPO_ISO_DESDE:
        # Diccionario DANE 2011-2024
        df['iso3'] = cod_num.map(
            lambda x: CODIGOS_PAIS_IMPO.get(int(x), None) if pd.notna(x) else None
        )
    else:
        # pycountry ISO 3166-1 numerico 2025+
        _cache = {}
        def _iso_lookup(x):
            if pd.isna(x): return None
            key = str(int(x)).zfill(3)
            if key not in _cache:
                try:
                    _cache[key] = pycountry.countries.get(numeric=key).alpha_3
                except Exception:
                    _cache[key] = None
            return _cache[key]
        df['iso3'] = cod_num.map(_iso_lookup)
    
    # Valor CIF
    df['cif_usd'] = limpiar_valor_impo(df['cif_usd'])
    
    # Filtrar
    mask = (
        df['cif_usd'].notna() & (df['cif_usd'] > 0) &
        df['iso3'].notna() &
        df['anio'].between(ANIOS[0], ANIOS[-1]) &
        df['mes'].between(1, 12)
    )
    cols = ['anio', 'mes', 'iso3', 'cif_usd', 'subpartida', 'kg_neto']
    return df.loc[mask, [c for c in cols if c in df.columns]].copy()


print('Funciones de limpieza listas')

Funciones de limpieza listas


---
## Sección 8 — Procesamiento año por año
Núcleo del pipeline V3. Procesa un año a la vez, guarda Parquet y libera RAM.  
Si el Parquet ya existe, lo omite. Para re-procesar: borrar el .parquet correspondiente.

In [103]:
# Funcion helper para guardar parquet con fallback
def guardar_parquet(df, path):
    """Guarda como CSV directamente para evitar incompatibilidad pyarrow."""
    csv_path = path.with_suffix('.csv')
    df.to_csv(csv_path, index=False)

sample_tag = f'_sample{SAMPLE_N}' if SAMPLE_N else ''

for anio in ANIOS:
    print(f'\n{"="*60}')
    print(f'ANIO {anio}')
    print(f'{"="*60}')
    
    # -- EXPORTACIONES --
    pq_expo = CLEAN_DIR / f'expo_{anio}{sample_tag}.parquet'
    if pq_expo.exists() or pq_expo.with_suffix('.csv').exists():
        print(f'  EXPO {anio}: cache existente, omitido')
    else:
        partes = archivos['expo'].get(anio, [])
        if not partes:
            print(f'  EXPO {anio}: no encontrado')
        else:
            frames = []
            for ruta in sorted(partes):
                frames.extend(leer_zip_expo(ruta))
            if frames:
                df_raw = pd.concat(frames, ignore_index=True)
                df_clean = limpiar_expo(df_raw, anio)
                guardar_parquet(df_clean, pq_expo)
                print(f'  EXPO {anio}: {len(df_raw):>10,} brutas -> {len(df_clean):>10,} limpias')
                top3 = df_clean.groupby('iso3')['fob_usd'].sum().nlargest(3)
                print(f'    Top 3 destinos: {dict(top3.items())}')
                del df_raw, df_clean
            else:
                print(f'  EXPO {anio}: sin datos legibles')
    
    # -- IMPORTACIONES --
    pq_impo = CLEAN_DIR / f'impo_{anio}{sample_tag}.parquet'
    if pq_impo.exists() or pq_impo.with_suffix('.csv').exists():
        print(f'  IMPO {anio}: cache existente, omitido')
    else:
        partes = archivos['impo'].get(anio, [])
        if not partes:
            print(f'  IMPO {anio}: no encontrado')
        else:
            frames = []
            for ruta in sorted(partes):
                frames.extend(leer_zip_impo(ruta))
            if frames:
                df_raw = pd.concat(frames, ignore_index=True)
                df_clean = limpiar_impo(df_raw, anio)
                guardar_parquet(df_clean, pq_impo)
                print(f'  IMPO {anio}: {len(df_raw):>10,} brutas -> {len(df_clean):>10,} limpias')
                usa_count = (df_clean['iso3'] == 'USA').sum()
                print(f'    Registros USA: {usa_count:,}')
                top3 = df_clean.groupby('iso3')['cif_usd'].sum().nlargest(3)
                print(f'    Top 3 origenes: {dict(top3.items())}')
                del df_raw, df_clean
            else:
                print(f'  IMPO {anio}: sin datos legibles')
    
    gc.collect()

print(f'\nProcesamiento completo. Parquets en: {CLEAN_DIR}')


ANIO 2011
  EXPO 2011:    411,377 brutas ->    411,372 limpias
    Top 3 destinos: {'USA': 21969134250.54, 'NLD': 2524104462.01, 'CHL': 2205006467.7599998}


KeyboardInterrupt: 

---
## Sección 9 — Carga desde Parquet
Punto de entrada rápido para ejecuciones subsecuentes (ejecutar S1-S5 y luego aquí).

In [ ]:
sample_tag = f'_sample{SAMPLE_N}' if SAMPLE_N else ''

def cargar_archivos_limpios(prefijo):
    frames = []
    stems = set()
    for f in CLEAN_DIR.iterdir():
        if f.name.startswith(prefijo) and sample_tag in f.name:
            stems.add(f.stem)
    
    for stem in sorted(stems):
        pq = CLEAN_DIR / f'{stem}.parquet'
        csv_f = CLEAN_DIR / f'{stem}.csv'
        loaded = False
        
        if pq.exists():
            try:
                frames.append(pd.read_parquet(pq))
                loaded = True
            except Exception:
                print(f'  Parquet corrupto: {pq.name}, intentando CSV...')
        
        if not loaded and csv_f.exists():
            frames.append(pd.read_csv(csv_f))
            print(f'  Cargado desde CSV: {csv_f.name}')
            loaded = True
        
        if not loaded:
            print(f'  SIN DATOS: {stem}')
    
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

df_expo = cargar_archivos_limpios('expo_')
df_impo = cargar_archivos_limpios('impo_')

print(f'\nExportaciones: {len(df_expo):>12,} filas | Anios: {sorted(df_expo["anio"].unique().tolist())}')
print(f'Importaciones: {len(df_impo):>12,} filas | Anios: {sorted(df_impo["anio"].unique().tolist())}')
print(f'\nPaises unicos EXPO: {df_expo["iso3"].nunique()}')
print(f'Paises unicos IMPO: {df_impo["iso3"].nunique()}')

  Cargado desde CSV: expo_2011.csv


C:\Users\camil\AppData\Local\Temp\ipykernel_35028\2396522211.py:23: DtypeWarning: Columns (0: kg_neto) have mixed types. Specify dtype option on import or set low_memory=False.
  frames.append(pd.read_csv(csv_f))


  Cargado desde CSV: expo_2012.csv


C:\Users\camil\AppData\Local\Temp\ipykernel_35028\2396522211.py:23: DtypeWarning: Columns (0: kg_neto) have mixed types. Specify dtype option on import or set low_memory=False.
  frames.append(pd.read_csv(csv_f))


  Cargado desde CSV: expo_2013.csv
  Cargado desde CSV: expo_2014.csv


...

  Cargado desde CSV: expo_2022.csv
  Cargado desde CSV: expo_2023.csv
  Cargado desde CSV: expo_2024.csv
  Cargado desde CSV: expo_2025.csv
  Cargado desde CSV: impo_2011.csv
  Cargado desde CSV: impo_2012.csv
  Cargado desde CSV: impo_2013.csv
  Cargado desde CSV: impo_2014.csv
  Cargado desde CSV: impo_2015.csv
  Cargado desde CSV: impo_2016.csv
  Cargado desde CSV: impo_2017.csv


C:\Users\camil\AppData\Local\Temp\ipykernel_35028\2396522211.py:23: DtypeWarning: Columns (0: kg_neto) have mixed types. Specify dtype option on import or set low_memory=False.
  frames.append(pd.read_csv(csv_f))


  Cargado desde CSV: impo_2018.csv
  Cargado desde CSV: impo_2019.csv
  Cargado desde CSV: impo_2020.csv
  Cargado desde CSV: impo_2021.csv
  Cargado desde CSV: impo_2022.csv


C:\Users\camil\AppData\Local\Temp\ipykernel_35028\2396522211.py:23: DtypeWarning: Columns (0: subpartida, 1: kg_neto) have mixed types. Specify dtype option on import or set low_memory=False.
  frames.append(pd.read_csv(csv_f))


  Cargado desde CSV: impo_2023.csv
  Cargado desde CSV: impo_2024.csv
  Cargado desde CSV: impo_2025.csv

Exportaciones:    6,835,866 filas | Anios: [2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Importaciones:   48,117,040 filas | Anios: [2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

Paises unicos EXPO: 225
Paises unicos IMPO: 231


---
## Sección 10 — Diagnóstico de datos
Validaciones para detectar problemas de mapeo.

In [ ]:
# -- Validar USA consistentemente --
print('=== VALIDACION USA ===')
for tipo, df, col_val in [('EXPO', df_expo, 'fob_usd'), ('IMPO', df_impo, 'cif_usd')]:
    print(f'\n{tipo}:')
    usa = df[df['iso3'] == 'USA'].groupby('anio')[col_val].sum()
    for anio, val in usa.items():
        print(f'  {anio}: USD {val/1e6:>10,.1f} M')
    if usa.empty:
        print('  SIN REGISTROS USA - revisar mapeo de paises!')

=== VALIDACION USA ===

EXPO:
  2011: USD   21,969.1 M
  2012: USD   21,833.3 M
  2013: USD   18,461.6 M
  2014: USD   14,223.7 M
  2015: USD   10,008.3 M
  2016: USD   10,215.3 M
  2017: USD   10,615.2 M
  2018: USD   10,674.3 M
  2019: USD   11,520.1 M
  2020: USD    8,921.8 M
  2021: USD   10,959.7 M
  2022: USD   14,840.2 M
  2023: USD   13,286.0 M
  2024: USD   14,335.0 M
  2025: USD   14,868.3 M

IMPO:
  2011: USD   13,548.9 M
  2012: USD   14,178.3 M
  2013: USD   16,336.7 M
  2014: USD   18,192.6 M
  2015: USD   15,512.4 M
  2016: USD   11,877.9 M
  2017: USD   12,014.5 M
  2018: USD   12,986.0 M
  2019: USD   13,276.8 M
  2020: USD   10,539.8 M
  2021: USD   14,071.3 M
  2022: USD   20,466.6 M
  2023: USD   15,997.7 M
  2024: USD   16,464.6 M
  2025: USD   16,175.5 M


In [ ]:
# -- Top 10 paises --
print('=== TOP 10 PAISES ===')
print('\nEXPO (FOB USD):')
top_expo = df_expo.groupby('iso3')['fob_usd'].sum().nlargest(10)
for pais, val in top_expo.items():
    print(f'  {pais}: USD {val/1e6:>12,.1f} M')

print('\nIMPO (CIF USD):')
top_impo = df_impo.groupby('iso3')['cif_usd'].sum().nlargest(10)
for pais, val in top_impo.items():
    print(f'  {pais}: USD {val/1e6:>12,.1f} M')

=== TOP 10 PAISES ===

EXPO (FOB USD):
  USA: USD    206,732.0 M
  PAN: USD     46,122.3 M
  CHN: USD     45,868.4 M
  ECU: USD     26,519.2 M
  NLD: USD     25,751.1 M
  BRA: USD     23,737.8 M
  IND: USD     22,354.6 M
  ESP: USD     21,628.3 M
  MEX: USD     19,297.0 M
  CHL: USD     18,094.6 M

IMPO (CIF USD):
  USA: USD    221,639.6 M
  CHN: USD    183,349.1 M
  MEX: USD     62,943.6 M
  BRA: USD     45,885.8 M
  DEU: USD     32,429.8 M
  FRA: USD     21,357.4 M
  JPN: USD     20,403.4 M
  IND: USD     19,061.5 M
  ARG: USD     17,418.1 M
  KOR: USD     15,722.2 M


In [ ]:
# -- Cobertura mensual --
print('=== COBERTURA MENSUAL ===')
for tipo, df in [('EXPO', df_expo), ('IMPO', df_impo)]:
    print(f'\n{tipo}:')
    cobertura = df.groupby('anio')['mes'].nunique()
    for anio, n_meses in cobertura.items():
        flag = '' if n_meses == 12 else f' (solo {n_meses} meses)'
        print(f'  {anio}: {n_meses} meses{flag}')

=== COBERTURA MENSUAL ===

EXPO:
  2011: 12 meses
  2012: 12 meses
  2013: 12 meses
  2014: 12 meses
  2015: 12 meses
  2016: 12 meses
  2017: 12 meses
  2018: 12 meses
  2019: 12 meses
  2020: 12 meses
  2021: 12 meses
  2022: 12 meses
  2023: 12 meses
  2024: 12 meses
  2025: 12 meses

IMPO:
  2011: 12 meses
  2012: 12 meses
  2013: 12 meses
  2014: 12 meses
  2015: 12 meses
  2016: 12 meses
  2017: 12 meses
  2018: 12 meses
  2019: 12 meses
  2020: 12 meses
  2021: 12 meses
  2022: 12 meses
  2023: 12 meses
  2024: 12 meses
  2025: 12 meses


In [ ]:
# -- Codigos no mapeados en IMPO (diagnostico para completar diccionario) --
print('=== CODIGOS IMPO SIN MAPEAR (por anio) ===')
for anio in ANIOS:
    # Verificar si existe archivo limpio (parquet o csv)
    pq = CLEAN_DIR / f'impo_{anio}{sample_tag}.parquet'
    csv_f = CLEAN_DIR / f'impo_{anio}{sample_tag}.csv'
    if not pq.exists() and not csv_f.exists():
        continue
    
    partes = archivos['impo'].get(anio, [])
    if not partes:
        continue
    frames = []
    for ruta in sorted(partes):
        frames.extend(leer_zip_impo(ruta))
    if not frames:
        continue
    df_raw = pd.concat(frames, ignore_index=True)
    if 'pais' not in df_raw.columns:
        del df_raw, frames
        continue
    cod_num = pd.to_numeric(df_raw['pais'], errors='coerce').dropna().astype(int)
    
    if anio < IMPO_ISO_DESDE:
        no_mapeados = set(cod_num.unique()) - set(CODIGOS_PAIS_IMPO.keys())
        no_mapeados = {c for c in no_mapeados if c < 900}
    else:
        no_mapeados = set()
        for c in cod_num.unique():
            key = str(c).zfill(3)
            try:
                pycountry.countries.get(numeric=key)
            except Exception:
                if c < 900:
                    no_mapeados.add(c)
    
    if no_mapeados:
        counts = cod_num[cod_num.isin(no_mapeados)].value_counts().head(10)
        print(f'\n  {anio}: {len(no_mapeados)} codigos sin mapear')
        for cod, cnt in counts.items():
            print(f'    cod {cod}: {cnt} filas')
    else:
        print(f'  {anio}: todos mapeados OK')
    del df_raw, frames
    gc.collect()

=== CODIGOS IMPO SIN MAPEAR (por anio) ===

  2011: 3 codigos sin mapear
    cod 885: 61 filas
    cod 688: 18 filas
    cod 579: 7 filas

  2012: 4 codigos sin mapear
    cod 688: 154 filas
    cod 885: 35 filas
    cod 888: 9 filas
    cod 579: 8 filas

  2013: 5 codigos sin mapear
    cod 688: 184 filas
    cod 885: 45 filas
    cod 579: 9 filas
    cod 130: 4 filas
    cod 888: 3 filas

  2014: 6 codigos sin mapear
    cod 688: 201 filas
    cod 130: 58 filas
    cod 885: 40 filas
    cod 579: 15 filas
    cod 626: 1 filas
    cod 888: 1 filas

  2015: 4 codigos sin mapear
    cod 633: 1415 filas
    cod 200: 60 filas
    cod 579: 15 filas
    cod 130: 4 filas

  2016: 7 codigos sin mapear
    cod 633: 3288 filas
    cod 200: 49 filas
    cod 130: 15 filas
    cod 579: 5 filas
    cod 636: 5 filas
    cod 621: 2 filas
    cod 624: 1 filas

  2017: 8 codigos sin mapear
    cod 633: 2729 filas
    cod 130: 317 filas
    cod 200: 15 filas
    cod 579: 6 filas
    cod 621: 4 filas
    

---
## Sección 11 — Exportación de JSON para D3.js
Genera los 3 archivos JSON que consume el sitio web.

In [ ]:
# VIZ 1: Mapa coropletico - FOB por pais destino y anio
viz1 = (
    df_expo
    .groupby(['anio', 'iso3'], as_index=False)['fob_usd']
    .sum()
    .assign(fob_musd=lambda d: (d['fob_usd'] / 1e6).round(2))
    .drop(columns='fob_usd')
    .rename(columns={'anio': 'año'})
    .sort_values(['año', 'fob_musd'], ascending=[True, False])
)
(OUTPUT_DIR / 'viz1_mapa_destinos.json').write_text(
    json.dumps(viz1.to_dict(orient='records'), ensure_ascii=False, indent=2),
    encoding='utf-8'
)
print(f'viz1_mapa_destinos.json: {len(viz1):,} registros | {viz1["iso3"].nunique()} paises')
print(f'  Ejemplo: {viz1.head(3).to_dict("records")}')

viz1_mapa_destinos.json: 2,811 registros | 225 paises
  Ejemplo: [{'año': 2011, 'iso3': 'USA', 'fob_musd': 21969.13}, {'año': 2011, 'iso3': 'NLD', 'fob_musd': 2524.1}, {'año': 2011, 'iso3': 'CHL', 'fob_musd': 2205.01}]


In [ ]:
# VIZ 2: Balanza comercial mensual
expo_m = df_expo.groupby(['anio','mes'])['fob_usd'].sum().reset_index(name='expo_usd')
impo_m = df_impo.groupby(['anio','mes'])['cif_usd'].sum().reset_index(name='impo_usd')

viz2 = (
    pd.merge(expo_m, impo_m, on=['anio','mes'], how='outer')
    .fillna(0)
    .assign(
        periodo            = lambda d: d['anio'].astype(str) + '-' + d['mes'].astype(str).str.zfill(2),
        exportaciones_musd = lambda d: (d['expo_usd'] / 1e6).round(2),
        importaciones_musd = lambda d: (d['impo_usd'] / 1e6).round(2),
        balanza_musd       = lambda d: ((d['expo_usd'] - d['impo_usd']) / 1e6).round(2),
    )
    .sort_values('periodo')
    .rename(columns={'anio': 'año'})
    [['periodo','año','mes','exportaciones_musd','importaciones_musd','balanza_musd']]
)
(OUTPUT_DIR / 'viz2_balanza_temporal.json').write_text(
    json.dumps(viz2.to_dict(orient='records'), ensure_ascii=False, indent=2),
    encoding='utf-8'
)
print(f'viz2_balanza_temporal.json: {len(viz2):,} periodos ({viz2["periodo"].min()} -> {viz2["periodo"].max()})')

viz2_balanza_temporal.json: 180 periodos (2011-01 -> 2025-12)


In [ ]:
# VIZ 3: Treemap sectores OMC
sector_anual = (
    df_expo
    .groupby(['anio','sector'])['fob_usd']
    .sum()
    .reset_index()
    .assign(fob_musd=lambda d: (d['fob_usd'] / 1e6).round(2))
)
total_anio = sector_anual.groupby('anio')['fob_musd'].transform('sum')
sector_anual['participacion_pct'] = (sector_anual['fob_musd'] / total_anio * 100).round(1)

anios_disp = sorted(sector_anual['anio'].unique().tolist())
viz3 = {
    'años_disponibles': [int(a) for a in anios_disp],
    'datos_por_año': {
        str(int(anio)): {
            'name': f'Exportaciones {int(anio)}',
            'children': [
                {
                    'name': r['sector'],
                    'value': float(r['fob_musd']),
                    'participacion_pct': float(r['participacion_pct']),
                }
                for _, r in g.sort_values('fob_musd', ascending=False).iterrows()
            ]
        }
        for anio, g in sector_anual.groupby('anio')
    }
}
(OUTPUT_DIR / 'viz3_treemap_sectores.json').write_text(
    json.dumps(viz3, ensure_ascii=False, indent=2),
    encoding='utf-8'
)
print(f'viz3_treemap_sectores.json: {len(anios_disp)} anios')
for a in anios_disp[:3]:
    children = viz3['datos_por_año'][str(int(a))]['children']
    print(f'  {int(a)}: {[(c["name"], c["participacion_pct"]) for c in children]}')

viz3_treemap_sectores.json: 15 anios
  2011: [('Combustibles e industrias extractivas', 65.1), ('Manufacturas', 31.3), ('Agropecuarios, alimentos y bebidas', 3.6)]
  2012: [('Combustibles e industrias extractivas', 65.7), ('Manufacturas', 30.5), ('Agropecuarios, alimentos y bebidas', 3.8)]
  2013: [('Combustibles e industrias extractivas', 66.8), ('Manufacturas', 29.3), ('Agropecuarios, alimentos y bebidas', 3.8)]


---
## Sección 12 — Push a GitHub (solo Colab)

In [ ]:
if MODO == 'colab':
    from google.colab import userdata
    try:
        GITHUB_TOKEN = userdata.get('GIT_TOKEN')
    except Exception:
        GITHUB_TOKEN = ''
        print('Token no configurado en Colab Secrets')
    
    if GITHUB_TOKEN:
        REPO_URL_AUTH = f'https://{GITHUB_TOKEN}@github.com/CamiloJose90/Exporta_Co.git'
        cmds = [
            ['git', '-C', str(REPO_DIR), 'config', 'user.email', 'pipeline@exporta.co'],
            ['git', '-C', str(REPO_DIR), 'config', 'user.name', 'Exporta Pipeline'],
            ['git', '-C', str(REPO_DIR), 'remote', 'set-url', 'origin', REPO_URL_AUTH],
            ['git', '-C', str(REPO_DIR), 'add', 'data/'],
            ['git', '-C', str(REPO_DIR), 'commit', '-m', f'data: actualizar JSONs V3{sample_tag}'],
            ['git', '-C', str(REPO_DIR), 'push'],
        ]
        for cmd in cmds:
            result = subprocess.run(cmd, capture_output=True, text=True)
            if result.returncode != 0 and 'nothing to commit' not in result.stdout:
                print(f'Error: {" ".join(cmd[3:])}')
                print(result.stderr)
                break
        else:
            print('JSONs pusheados a GitHub')
else:
    print('Push omitido en modo local.')

Push omitido en modo local.
